# **Third Trial**

> Timer: 17:15

In [74]:
import pandas as pd
import pickle
import os
import random
import spacy
from string import punctuation
import random

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Utility**

In [75]:
lemmatizerx = WordNetLemmatizer()
stopx = stopwords.words('english')
categories = {}

In [76]:
data = pd.read_csv('./jobpostingdata.csv')
X = data['text']
Y = data['fraudulent']

data.head()

,title,fraudulent,text
0,PHP Developer,0,PHP Developer You're a skilled developer. You ...
1,CUSTOMER SERVICE AGENT,1,CUSTOMER SERVICE AGENT Aegis is a global busi...
2,VP Marketing & Growth,0,VP Marketing & Growth Depop is an exciting new...
3,SAP BW Developer/Architect,1,SAP BW Developer/Architect Assist with Full L...
4,Administrative Assistant,1,Administrative Assistant With decades of exper...


### **Preprocessing**

> Timer Pause: 17:22

> Timer Continue: 17:40

In [77]:
def AlterTag (tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    return 'n'

def Preprocessing (docx: str):
    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok not in stopx]
    tokens = [tok for tok in tokens if tok.isalpha()]

    tagged = pos_tag(tokens)
    tokens = [lemmatizerx.lemmatize(tok, AlterTag(tag)) for tok, tag in tagged]
    return tokens

### **Training**

In [78]:
def Train ():
    print('Start Training...')
    print('')
    # Feats
    feats = []

    all_tokens = Preprocessing(' '.join(X))
    freqx = FreqDist(all_tokens)
    print('Most Common Feature:')
    print(freqx.most_common(5))
    print('')

    for sent, label in zip(X,Y):
        clean = Preprocessing(sent)
        ft = {word: True for word in clean}
        feats.append([ft, label])

    # Split
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]

    # Train
    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    # Info
    print('Model Trained')
    print(f'Accuracy: {acc*100}%')
    print('')

    # Save
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    return model

def Load ():
    print('Loading Model...')
    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
    else:
        print('Model Not Found')
        model = Train()
    return model

### **Encoding Models**

> Timer Pause: 17:55

> Timer Continue: 07:11

In [79]:
def TF_IDF (query: str):
    vectorizer = TfidfVectorizer(stop_words='english')
    mtx = vectorizer.fit_transform(X)
    qvec = vectorizer.transform([query])

    sim = cosine_similarity(mtx, qvec)
    data['Similarity'] = sim

    sorted_data = data.sort_values(by='Similarity', ascending=False)
    print('Top 5 Job Recommendation for You')
    for i in range (5):
        print(f'{i+1}. {sorted_data.iloc[i,0]}')

def Ngrams (query: str):
    vectorizer = TfidfVectorizer(ngram_range=(1,3), stop_words='english')
    mtx = vectorizer.fit_transform(X)
    qvec = vectorizer.transform([query])

    sim = cosine_similarity(mtx, qvec)
    data['Similarity'] = sim

    sorted_data = data.sort_values(by='Similarity', ascending=False)
    print('Top 5 Job Recommendation for You')
    for i in range (5):
        print(f'{i+1}. {sorted_data.iloc[i,0]}')

### **NER**

In [80]:
def InitNER ():
    ner = spacy.load('en_core_web_sm')
    parg = ' '.join(data['text'].head(100))
    res = ner(parg)

    for ent in res.ents:
        label = ent.label_

        if label not in categories:
            categories[label] = []
        categories[label].append(ent.text)
            

### **Support Function**

In [81]:
textx = None
categorex = None

In [ ]:
def WriteText (model):
    global textx, categorex

    print('Please Enter Your Text: ')
    txt = input("")

    if len(txt) < 20 or len(txt.split(' ')) < 3:
        print('Please Input at Least 3 Words and 20 Characters')
        return None
    
    textx = txt

    clean = Preprocessing(textx)
    feats = {word: True for word in clean}
    categorex = model.classify(feats)
    categorex = "Fraud" if categorex == 1 else "Normal"

    print(f'Your Text: {textx}')
    print(f'Your Text Category: {categorex}')

def ViewRecomx ():
    if textx == None:
        print('(!) Please input your Text First')
        return None
    
    print('Choose Embedding Model:')
    print('1. TF-IDF')
    print('2. NGrams')
    print('>> ')
    cc = input()

    if cc == '1':
        TF_IDF(textx)
    elif cc == '2':
        Ngrams(textx)
    else:
        print('Invalid Input')

def ViewNER ():
    if textx == None:
        print('(!) Please Input your Text First')
        return None

    ner = spacy.load('en_core_web_sm')
    res = ner(textx)
    print('Named Entity Recognition')
    for ent in res.ents:
        print(f'{ent.label_}: {ent.text}')
    # if not categories:
    #     print('Initializing NER...')
    #     InitNER()

    # for label, txt in categories.items():

    #     print(f'{label}: {txt}')



### **Main Function**

In [83]:
def Menu ():
    model = Load()

    while True:
        print('')
        print(f'Your Text: {textx}')
        print(f'Your Text Category: {categorex}')
        print('===')
        print('1. Write your Text')
        print('2. View Job Recommendation')
        print('3. View NER')
        print('4. Exit')
        print('>> ')
        cc = input()
        print('')
        if cc == '1':
            WriteText(model)
        elif cc == '2':
            ViewRecomx()
        elif cc == '3':
            ViewNER()
        elif cc == '4':
            return
        else:
            print('Invalid Input')
        

In [84]:
Menu()

Loading Model...

Your Text: None
Your Text Category: None
===
1. Write your Text
2. View Job Recommendation
3. View NER
4. Exit
>> 

Please Enter Your Text: 
Your Text: Ken love Software Engineering by Ms Andien
Your Text Category: Normal

Your Text: Ken love Software Engineering by Ms Andien
Your Text Category: Normal
===
1. Write your Text
2. View Job Recommendation
3. View NER
4. Exit
>> 

Initializing NER...
ORG: ['PHP Developer', 'Computer Science, Electronic Engineering', 'PHP', 'required)We’ll', 'Healthcare', 'Travel', 'Hospitality, Consumer Goods', 'Essar', 'TX', 'DescriptionRepresentative', 'Customer Service - TX - Dallas Customer Service', 'the Customer Service Representative', 'The Customer Service Representative', 'Accurate', 'Customer Service', 'DENTAL INSURANCE', 'VP Marketing & Growth Depop', 'VC’s', 'Sell &amp', 'Community', 'Product, Operations, Development', 'SEM', 'DepopIt’s', 'Old StreetApple', 'SAP BW Developer/Architect  Assist', 'Full Lifecycle BW Implementation

In [85]:
# Ken love Software Engineering by Ms Andien